In [1]:
# Standard library imports
from collections import deque
from glob import glob
import itertools
import json
import logging
import math
import os
import random
import re
import time
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

# Third-party imports
import langid
import numpy as np
import pyarrow as pa
import pyarrow.dataset as ds
import tiktoken
import torch
import torch.nn as nn
from pynvml import nvmlDeviceGetHandleByIndex, nvmlDeviceGetMemoryInfo, nvmlInit
from torch.amp import GradScaler, autocast
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, IterableDataset, get_worker_info
from tqdm import tqdm


In [2]:

def show_vram_usage():
    """
    Show GPU memory usage in the current moment.
    """
    nvmlInit()
    handle = nvmlDeviceGetHandleByIndex(0)
    info = nvmlDeviceGetMemoryInfo(handle)
    return str(f"Used: {info.used // (1024 ** 2)} MB, Free: {info.free // (1024 ** 2)} MB, Total: {info.total // (1024 ** 2)} MB")

In [3]:
print(show_vram_usage())

Used: 603 MB, Free: 11684 MB, Total: 12288 MB


In [4]:

import math
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


# ============================================================
# RMSNorm (Root Mean Square Normalization, Gemma/LLaMA style)
# ============================================================
class RMSNorm(nn.Module):
    """
    Root Mean Square Normalization (RMSNorm).

    Compared to LayerNorm, RMSNorm:
    - Normalizes using the root mean square of activations
      (ignores mean subtraction, only rescales variance).
    - Reduces computational overhead and avoids shifting mean,
      which improves numerical stability in large transformers.
    - Used in Gemma, LLaMA, Falcon, etc.

    Differences from LayerNorm:
    - No centering step (mean subtraction removed).
    - Scale parameter is zero-centered and applied as (1 + weight).
    - Optional shift (bias) can be added.

    Why (1 + scale) instead of direct scaling?
    -----------------------------------------
    Using (1 + scale) with zero-initialized weights provides a "residual connection" 
    for the normalization process. During initialization, scale=0 means the layer 
    acts as identity function (x * (1+0) = x), which:
    1. Improves training stability in deep networks
    2. Allows the model to gradually learn appropriate scaling factors
    3. Matches the implementation in Gemma/LLaMA where weights start near identity

    This is superior to standard scaling (x * scale) which would zero-out activations 
    at initialization if scale=0.

    Args:
        dim (int): Feature dimension to normalize.
        eps (float, optional): Small constant to avoid divide-by-zero.
        bias (bool, optional): Whether to include a learnable shift.

    Returns:
        Tensor of the same shape, normalized along the last dimension.
    """
    def __init__(self, dim: int, eps: float = 1e-6, bias: bool = False):
        super().__init__()
        self.eps = eps
        # Gemma uses 0-centered weights and applies (1 + weight) at runtime
        self.scale = nn.Parameter(torch.zeros(dim))
        self.shift = nn.Parameter(torch.zeros(dim)) if bias else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_dtype = x.dtype
        x_f = x.float()                                      # normalize in float32
        var = x_f.pow(2).mean(dim=-1, keepdim=True)          # mean square
        x_norm = x_f * torch.rsqrt(var + self.eps)           # normalize
        out = x_norm * (1.0 + self.scale.float())            # scale (1 + w)
        if self.shift is not None:
            out = out + self.shift.float()                   # optional bias
        return out.to(input_dtype)


# ============================================================
# RoPE (Rotary Position Embeddings)
# ============================================================
def _build_rope_cache(head_dim: int, base: int, max_seq_len: int, device, dtype=torch.float32):
    """
    Precompute sine and cosine rotation matrices for RoPE.

    Args:
        head_dim (int): Per-head dimensionality (must be even).
        base (int): Exponential base for frequency spectrum (default 10k).
        max_seq_len (int): Maximum supported sequence length.
        device (torch.device): Device to place the cache.
        dtype (torch.dtype): Precision (float32 recommended for accuracy).

    Returns:
        (cos, sin): Two [T, head_dim] tensors with precomputed values.

    Why RoPE?
    ---------
    - Standard GPT-2 used learned absolute position embeddings.
    - RoPE encodes positions by rotating queries/keys in attention.
    - Enables extrapolation to longer contexts, and is cheaper to store.
    - Adopted in GPTNeoX, LLaMA, Gemma, and other modern LLMs.
    """
    assert head_dim % 2 == 0, "RoPE head_dim must be even"
    half = head_dim // 2
    inv_freq = 1.0 / (base ** (torch.arange(0, half, dtype=dtype, device=device) / half))
    t = torch.arange(max_seq_len, dtype=dtype, device=device)
    freqs = torch.outer(t, inv_freq)                       # [T, half]
    emb = torch.cat([freqs, freqs], dim=-1)                # [T, head_dim]
    return torch.cos(emb), torch.sin(emb)


def _apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    Apply rotary position embedding (RoPE) to queries/keys.

    RoPE Mathematical Explanation
    ----------------------------
    The rotation operation implements:
    
    [x1, x2] → [x1·cos(θ) - x2·sin(θ), x1·sin(θ) + x2·cos(θ)]
    
    Which is equivalent to complex multiplication:
    (x1 + i·x2) · e^(iθ) = (x1 + i·x2) · (cosθ + i·sinθ)
    
    This preserves the magnitude of the vector while encoding position:
    ||[x1·cosθ - x2·sinθ, x1·sinθ + x2·cosθ]|| = ||[x1, x2]||
    
    Key advantage over absolute positional embeddings:
    - Relative positional information is preserved in attention patterns
    - Enables extrapolation to longer sequences than trained on
    - No additional parameters needed (purely positional encoding)

    Args:
        x (Tensor): Shape [B, H, T, D] (batch, heads, tokens, head_dim).
        cos, sin (Tensor): RoPE caches of shape [T, D].

    Returns:
        Tensor with same shape, rotated according to RoPE.

    Notes:
        - Splits head_dim into two halves, applies 2D rotation.
        - Keeps dtype consistent with input x.
    """
    B, H, T, D = x.shape
    x1, x2 = x[..., : D // 2], x[..., D // 2:]
    cos = cos[:T, :].unsqueeze(0).unsqueeze(0)             # (1,1,T,D)
    sin = sin[:T, :].unsqueeze(0).unsqueeze(0)
    rot = torch.cat([-x2, x1], dim=-1)                     # rotated halves
    x_rot = x * cos + rot * sin
    return x_rot.to(dtype=x.dtype)


# ============================================================
# SwiGLU FeedForward (used in LLaMA, Gemma)
# ============================================================
class FeedForward(nn.Module):
    """
    SwiGLU MLP feed-forward layer (LLaMA/Gemma style).

    Compared to vanilla FFN:
    - Uses two parallel linear layers (gating + up projection).
    - Combines them with element-wise product after SiLU activation.
    - Improves expressivity and stability in LLM training.

    Why SwiGLU Outperforms Standard FFN?
    -----------------------------------
    SwiGLU (x·σ(xW₁+b₁))W₃ + b₃ offers three key advantages:

    1. **Higher Expressivity**: 
    The gating mechanism (σ(xW₁+b₁)) creates conditional pathways through the network,
    allowing more complex functions than ReLU(xW₁)W₂.

    2. **Smoother Optimization**:
    SiLU (swish) has non-zero gradients for negative inputs, avoiding "dying ReLU" problem.
    Combined with multiplicative gating, this creates smoother loss landscapes.

    3. **Empirical Performance**:
    LLaMA/Gemma teams observed ~10% faster convergence and better final perplexity
    compared to standard GeLU-based FFNs at scale.

    Note: The 4x hidden dimension maintains similar parameter count to standard FFN
    (3.5x would be more precise but 4x is standard for compatibility).

    Args:
        n_embd (int): Embedding dimension.
        dropout (float): Dropout probability.

    Returns:
        Tensor of same shape [B, T, C].
    """
    def __init__(self, n_embd: int, dropout: float):
        super().__init__()
        hidden = 4 * n_embd  # factor can be tuned (e.g., 3.5x)
        self.w1 = nn.Linear(n_embd, hidden, bias=False)  # gate
        self.w2 = nn.Linear(n_embd, hidden, bias=False)  # up
        self.w3 = nn.Linear(hidden, n_embd, bias=False)  # down
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))


# ============================================================
# Multi-Head Attention with RoPE + FlashAttention (SDPA)
# ============================================================
class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention with RoPE and fused QKV.

    Features:
    - Fused QKV projection (single Linear → 3 * n_embd).
    - Rotary embeddings applied to Q/K.
    - Uses PyTorch's scaled_dot_product_attention (FlashAttention).
    - Causal masking enabled (autoregressive).

    Args:
        n_embd (int): Embedding dimension.
        num_heads (int): Number of attention heads.
        block_size (int): Max context length.
        dropout (float): Dropout rate.
        qkv_bias (bool): Whether to include bias in QKV projections.
    """
    def __init__(self, n_embd: int, num_heads: int, block_size: int, dropout: float, qkv_bias: bool) -> None:
        super().__init__()
        assert n_embd % num_heads == 0, "n_embd must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_size = n_embd // num_heads
        self.dropout = dropout
        self.block_size = block_size

        # Projections
        self.qkv_proj = nn.Linear(n_embd, 3 * n_embd, bias=qkv_bias)
        self.out_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # RoPE caches (precomputed in float32 for stability)
        cos, sin = _build_rope_cache(
            head_dim=self.head_size,
            base=10_000,                 # standard RoPE base
            max_seq_len=block_size,
            device="cpu",                # moved later to correct device
            dtype=torch.float32,
        )
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        # This ensures buffers move with the model
        self._register_load_state_dict_pre_hook(self._load_state_dict_pre_hook)

    def _load_state_dict_pre_hook(self, state_dict, prefix, local_metadata, strict, 
                                 missing_keys, unexpected_keys, error_msgs):
        """Ensure RoPE buffers get properly loaded and moved to device."""
        cos_key = prefix + "rope_cos"
        sin_key = prefix + "rope_sin"
        if cos_key in state_dict and sin_key in state_dict:
            self.rope_cos = state_dict[cos_key]
            self.rope_sin = state_dict[sin_key]
    
    def to(self, *args, **kwargs):
        """Override to() to ensure RoPE cache moves with the model."""
        device, dtype, non_blocking, convert_to_format = \
            torch._C._nn._parse_to(*args, **kwargs)
        
        # Call parent's to() first
        super_result = super().to(*args, **kwargs)
        
        # Move RoPE buffers to the same device
        if device is not None:
            self.rope_cos = self.rope_cos.to(device=device)
            self.rope_sin = self.rope_sin.to(device=device)
            
        return super_result

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=2)

        q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2)

        # RoPE
        q = _apply_rope(q, self.rope_cos, self.rope_sin)
        k = _apply_rope(k, self.rope_cos, self.rope_sin)

        # FlashAttention
        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.attn_dropout.p if self.training else 0.0,
            is_causal=True,
            scale=self.head_size ** -0.5
        )

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


# ============================================================
# Transformer Block (Pre-Norm with RMSNorm + checkpointing)
# ============================================================
class Block(nn.Module):
    """
    Transformer block (Pre-Norm architecture).

    Structure:
    - Input → RMSNorm → MultiHeadAttention → Residual
    - Input → RMSNorm → SwiGLU FeedForward → Residual

    Differences from GPT-2:
    - Uses RMSNorm instead of LayerNorm (faster, stabler).
    - Pre-Norm (normalize before sublayers) improves gradient flow.
    - Supports checkpointing for memory-efficient training.

    Args:
        n_embd (int): Embedding dimension.
        n_head (int): Number of attention heads.
        block_size (int): Max sequence length.
        dropout (float): Dropout rate.
        qkv_bias (bool): Bias in QKV projections.
        use_checkpoint (bool): Enable torch.utils.checkpoint.
    """
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float, qkv_bias: bool, use_checkpoint: bool = False) -> None:
        super().__init__()
        self.use_checkpoint = use_checkpoint

        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        self.sa = MultiHeadAttention(n_embd, n_head, block_size, dropout, qkv_bias)
        self.ffwd = FeedForward(n_embd, dropout)

        self.ln1 = RMSNorm(n_embd, eps=1e-6)
        self.ln2 = RMSNorm(n_embd, eps=1e-6)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training and self.use_checkpoint:
            sa_out = checkpoint(self.sa, self.ln1(x))
        else:
            sa_out = self.sa(self.ln1(x))
        x = x + sa_out

        if self.training and self.use_checkpoint:
            ff_out = checkpoint(self.ffwd, self.ln2(x))
        else:
            ff_out = self.ffwd(self.ln2(x))
        x = x + ff_out
        return x



# ============================================================
# GPTLanguageModel (modernized GPT-style architecture)
# ============================================================
class GPTLanguageModel(nn.Module):
    """
    GPT-style autoregressive language model (modernized variant).

    This implementation preserves the public API of a standard GPT
    (init, forward, generate) but introduces several architectural
    improvements inspired by recent LLMs (Gemma, LLaMA, Falcon):

    Key Design Choices
    ------------------
    1. **RMSNorm (instead of LayerNorm)**:
       - Normalizes activations using root mean square (no mean subtraction).
       - Reduces compute cost, avoids mean-centering, and improves stability
         at large scale.
       - Parameters are zero-centered and applied as (1 + weight), matching
         Gemma/LLaMA conventions.

    2. **Rotary Position Embeddings (RoPE)**:
       - Replaces learned absolute positional embeddings.
       - Encodes positions via rotation in query/key space.
       - Enables efficient extrapolation to longer sequence lengths.
       - Memory-efficient (no large embedding table needed).

    3. **SwiGLU FeedForward (vs. vanilla FFN)**:
       - Parallel gating structure: SiLU(W1x) * (W2x).
       - Higher expressivity and smoother gradients.
       - Used in LLaMA/Gemma for improved convergence.

    4. **Pre-Norm Transformer Blocks**:
       - Apply normalization *before* attention/FFN sublayers.
       - Improves gradient flow in deep networks.
       - Now the de facto standard in modern architectures.

    5. **FlashAttention (via PyTorch SDPA)**:
       - Uses `scaled_dot_product_attention` with causal masking.
       - Highly optimized fused attention kernel.
       - Reduces memory usage and speeds up training.

    6. **Gradient Checkpointing Support**:
       - Optionally re-computes activations in forward pass.
       - Saves GPU memory at the cost of some compute.

    7. **Tied Embeddings**:
       - Output layer shares weights with input embedding table.
       - Reduces parameter count, improves sample efficiency.

    Public API
    ----------
    - `forward(input_tokens, targets=None)`:
        Returns `(logits, loss)` for next-token prediction.
    - `generate(input_tokens, max_new_tokens, **kwargs)`:
        Simple entry point for text generation.
    - `advanced_generation(...)`:
        Extended sampling strategies (temperature, top-k, top-p, EOS).

    Args:
        vocab_size (int): Vocabulary size.
        n_embd (int): Embedding dimension.
        n_head (int): Number of attention heads.
        block_size (int): Maximum sequence length (context window).
        n_layer (int): Number of transformer blocks.
        dropout (float): Dropout probability.
        device (str): Target device ("cuda" or "cpu").
        qkv_bias (bool, optional): Whether QKV projections use bias.
        ignore_index (int, optional): Index to ignore in loss.
        use_checkpoint (bool, optional): Enable gradient checkpointing.

    Example:
        >>> model = GPTLanguageModel(
        ...     vocab_size=50257, n_embd=768,
        ...     n_head=12, block_size=1024,
        ...     n_layer=12, dropout=0.1, device="cuda"
        ... )
        >>> logits, loss = model(input_ids, targets)
        >>> generated = model.generate(input_ids, max_new_tokens=50, top_k=40)

    Notes:
        - This model is API-compatible with GPT-2 style codebases.
        - Architectural updates improve convergence, stability,
          and efficiency on modern hardware.
    """
    def __init__(
        self,
        vocab_size: int,
        n_embd: int,
        n_head: int,
        block_size: int,
        n_layer: int,
        dropout: float,
        device: str,
        qkv_bias: bool = False,
        ignore_index: int = -100,
        use_checkpoint: bool = False,
    ) -> None:
        super().__init__()
        self.ignore_index = ignore_index
        self.block_size = block_size
        self.device = device
        self.n_layer = n_layer

        # Token embeddings only (RoPE replaces absolute position embeddings)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)

        # Transformer blocks
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout, qkv_bias, use_checkpoint=use_checkpoint)
            for _ in range(n_layer)
        ])

        # Final RMSNorm
        self.ln_f = RMSNorm(n_embd, eps=1e-6)

        # Output head (tied weights)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=True)
        self.lm_head.weight = self.token_embedding_table.weight

        # Init + move to device
        self.apply(self._init_weights)
        self.to(device)

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            # Scale initialization by layer depth (Fan-in initialization)
            std = 0.02 / math.sqrt(2 * layer_idx) if hasattr(module, 'layer_idx') else 0.02
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        input_tokens: torch.Tensor,
        targets: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B, T = input_tokens.shape
        # Crop sequences to block_size during training
        if T > self.block_size:
            x = x[:, -self.block_size:]
            if targets is not None:
                targets = targets[:, -self.block_size:]
                logger.warning(f"Sequence length {T} hac been cropped due to exceeding block_size {self.block_size}")

        # Token embeddings (no absolute pos_emb; RoPE is inside attention)
        x = self.token_embedding_table(input_tokens)  # (B,T,C)
        x = self.drop(x)

        # Blocks + final norm
        x = self.blocks(x)
        x = self.ln_f(x)

        # Logits
        logits = self.lm_head(x)  # (B,T,vocab)

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T), ignore_index=self.ignore_index)
        return logits, loss


    def generate(
        self, 
        input_tokens: torch.Tensor, 
        max_new_tokens: int, 
        **kwargs
    ) -> torch.Tensor:
        """
        Unified token generation interface (simple entry point).

        This method provides a convenient wrapper around
        :meth:`advanced_generation`, exposing the same generation
        interface as popular LLM frameworks.

        By default it calls :meth:`advanced_generation` with
        ``temperature=1.0`` and passes through any additional keyword
        arguments for advanced sampling.

        Args:
            input_tokens (torch.Tensor):
                Input tensor of shape ``(B, T)`` containing initial tokens
                (batch size *B*, context length *T*).
            max_new_tokens (int):
                Number of new tokens to generate and append to the input.
            **kwargs:
                Additional parameters forwarded to
                :meth:`advanced_generation` (e.g. ``temperature``,
                ``top_k``, ``top_p``, ``eos_token_id``).

        Returns:
            torch.Tensor:
                Extended token sequence of shape ``(B, T + max_new_tokens)``.

        Example:
            >>> model.generate(start_tokens, max_new_tokens=50, top_k=40)
        """
        return self.advanced_generation(
            input_tokens,
            max_new_tokens,
            temperature=1.0,
            **kwargs,
        )

    @torch.no_grad()
    def advanced_generation(
        self,
        input_tokens: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
        top_p: Optional[float] = None,
        eos_token_id: int = 50256,
    ) -> torch.Tensor:
        """
        Autoregressive text generation with flexible sampling strategies.

        Implements incremental token-by-token decoding for GPT-style
        language models, supporting greedy decoding, top-*k* filtering,
        nucleus (top-*p*) sampling, and EOS-based early stopping.

        Args:
            input_tokens (torch.Tensor):
                Initial token IDs of shape ``(B, T)``, where *B* is the
                batch size and *T* is the sequence length.
            max_new_tokens (int):
                Maximum number of new tokens to append to the input.
            temperature (float, optional, default=1.0):
                Scaling factor applied to logits before softmax.
                - ``temperature=0.0`` → deterministic greedy decoding.
                - ``temperature>1.0`` → more random sampling.
                - ``0.0<temperature<1.0`` → sharper, less random outputs.
            top_k (int, optional):
                Restrict sampling to the *k* most likely tokens at each
                step (default: disabled).
            top_p (float, optional):
                Restrict sampling to the smallest token set whose
                cumulative probability exceeds *p* (default: disabled).
                Must satisfy ``0 < top_p <= 1``.
            eos_token_id (int, optional, default=50256):
                Stop generation if this token is encountered.

        Returns:
            torch.Tensor:
                Generated sequence of shape ``(B, T + max_new_tokens)``
                including the original prompt and newly generated tokens.

        Notes:
            - Supports batch generation with different prompts.
            - Applies block-size cropping internally to respect model's
              maximum context length.
            - When both ``top_k`` and ``top_p`` are enabled, filtering
              is applied sequentially (top-*k* first, then top-*p*).

        Example:
            >>> model.advanced_generation(
            ...     start_tokens,
            ...     max_new_tokens=128,
            ...     temperature=0.8,
            ...     top_k=50,
            ...     top_p=0.9,
            ... )
        """
        if temperature < 0:
            raise ValueError("temperature must be >= 0")
        if top_k is not None and top_k <= 0:
            raise ValueError("top_k must be a positive integer or None")
        if top_p is not None and not (0 < top_p <= 1):
            raise ValueError("top_p must be in (0, 1]")

        for _ in range(max_new_tokens):
            # Crop sequence to block size for efficiency
            cropped_input = input_tokens[:, -self.block_size:]

            # Forward pass (no loss)
            logits, _ = self(cropped_input)

            # Focus only on last position logits
            next_logits = logits[:, -1, :]

            # Greedy decoding shortcut
            if temperature == 0.0:
                next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
                input_tokens = torch.cat((input_tokens, next_token), dim=1)
                if next_token.item() == eos_token_id:
                    break
                continue

            # Apply temperature scaling
            scaled_logits = next_logits / temperature

            # Apply top-k filtering
            if top_k is not None:
                k = min(top_k, scaled_logits.size(-1))
                top_vals = torch.topk(scaled_logits, k, dim=-1).values
                scaled_logits = scaled_logits.masked_fill(
                    scaled_logits < top_vals[:, [-1]], -float("inf")
                )

            # Convert to probabilities
            probs = F.softmax(scaled_logits, dim=-1)

            # Apply nucleus (top-p) filtering
            if top_p is not None:
                sorted_probs, sorted_idx = torch.sort(probs, dim=-1, descending=True)
                cumprobs = torch.cumsum(sorted_probs, dim=-1)
                to_remove = cumprobs > top_p
                to_remove[..., 1:] = to_remove[..., :-1].clone()
                to_remove[..., 0] = False
                remove_mask = torch.zeros_like(probs, dtype=torch.bool)
                remove_mask.scatter_(-1, sorted_idx, to_remove)
                probs = probs.masked_fill(remove_mask, 0.0)
                probs = probs / probs.sum(dim=-1, keepdim=True)

            # Sample token
            next_token = torch.multinomial(probs, num_samples=1)

            # Append to sequence
            input_tokens = torch.cat((input_tokens, next_token), dim=1)
            if eos_token_id is not None:
                # Check if ALL sequences have generated EOS (for batch=1 this is equivalent)
                if (next_token == eos_token_id).all():
                    break

        return input_tokens

In [5]:
print(show_vram_usage())

Used: 603 MB, Free: 11684 MB, Total: 12288 MB


In [6]:
print(show_vram_usage())

Used: 603 MB, Free: 11684 MB, Total: 12288 MB


In [7]:
# --- parquet -> tokens -> (x,y) streaming dataset -----------------

def choose_string_column(dataset: ds.dataset) -> str:
    schema = dataset.schema
    str_columns = [f.name for f in schema if pa.types.is_string(f.type)]
    for candidate in ("text","content","body"):
        if candidate in str_columns:
            return candidate
    if not str_columns:
        raise RuntimeError("No string column in parquet schema(s)")
    return str_columns[0]

def is_english_text(text: str, mode: str = "accuracy", ascii_threshold: float = 0.5) -> bool:
    if mode == "skip": return True
    if not text: return False
    def ascii_fraction(s): 
        return 0.0 if not s else sum(1 for c in s if ord(c) < 128)/len(s)
    if mode in ("ascii","fast"): return ascii_fraction(text) >= ascii_threshold
    try:
        lang, _ = langid.classify(text)
        if lang == "en": return True
    except Exception: pass
    return ascii_fraction(text) >= ascii_threshold

def _get_eos_token_id(enc: "tiktoken.Encoding", model_hint: str = "gpt2") -> Optional[int]:
    try:
        ids = enc.encode("", allowed_special={"<|endoftext|>"})
        if ids: return int(ids[0])
    except Exception: pass
    try:
        ids = enc.encode("")
        if ids: return int(ids[0])
    except Exception: pass
    if model_hint.lower().startswith("gpt2"):
        return 50256
    return None

class ParquetCausalIterable(IterableDataset):
    """
    Streams parquet rows -> filter -> tokenize -> rolling buffer -> emits (x,y)
    pairs of length `block_size` with stride `stride`.

    Notes
    -----
    - Designed for DataLoader(num_workers>0, prefetch_factor>1, persistent_workers=True).
    - Each worker scans its own shard of files (round-robin).
    - Tokenizer/langid are created inside worker processes.
    - EOS is appended after each *row*; crossing rows is allowed (continuous stream).
    """
    def __init__(
        self,
        files: Optional[List[str]] = None,
        input_dir: Optional[str] = None,
        pattern: str = "*.parquet",
        text_col: Optional[str] = None,
        batch_rows: int = 4096,
        target_chunk_bytes: int = 100_000_000,
        tokenizer_model: str = "gpt2",
        add_eos: bool = True,
        block_size: int = 1024,
        stride: Optional[int] = None,
        lang_mode: str = "skip",
        ascii_threshold: float = 0.5,
        shuffle_files: bool = True,
        seed: int = 1337,
    ):
        self._all_files = files
        self.input_dir = input_dir
        self.pattern = pattern
        self.text_col = text_col
        self.batch_rows = int(batch_rows)
        self.target_chunk_bytes = int(target_chunk_bytes)
        self.tokenizer_model = tokenizer_model
        self.add_eos = add_eos
        self.block_size = int(block_size)
        self.stride = int(stride if stride is not None else block_size)  # no overlap by default
        self.lang_mode = lang_mode
        self.ascii_threshold = ascii_threshold
        self.shuffle_files = shuffle_files
        self.seed = int(seed)

    def _files(self) -> List[str]:
        if self._all_files is not None:
            return list(self._all_files)
        if not self.input_dir:
            raise ValueError("Provide either `files` or `input_dir`.")
        p = Path(self.input_dir)
        if not p.exists():
            raise FileNotFoundError(self.input_dir)
        return sorted(str(x) for x in p.glob(self.pattern))

    def _shard_for_worker(self, files: List[str]) -> List[str]:
        info = get_worker_info()
        if info is None:
            shard = files
            rng = random.Random(self.seed)
        else:
            shard = files[info.id::info.num_workers]  # round-robin
            rng = random.Random(self.seed + info.id)
        if self.shuffle_files:
            rng.shuffle(shard)
        return shard

    def _yield_sequences(self, token_buffer: List[int]) -> Iterable[torch.Tensor]:
        """
        Given a growing token buffer, yield fixed-length (block_size+1) sequences
        advancing by self.stride. Uses an index pointer to avoid O(N) pops.
        """
        i = 0
        bs1 = self.block_size + 1
        # periodically compact buffer to control memory
        COMPACT_EVERY = max(8 * self.block_size, 32768)
        while i + bs1 <= len(token_buffer):
            seq = token_buffer[i : i + bs1]
            yield torch.tensor(seq, dtype=torch.long)  # CPU tensor; DataLoader can pin & move
            i += self.stride
            if i > COMPACT_EVERY:
                # drop consumed prefix
                del token_buffer[:i]
                i = 0

        # keep only the tail we still need for future continuation
        if i > 0:
            del token_buffer[:i]

    def __iter__(self):
        # Per-worker state
        enc = tiktoken.get_encoding(self.tokenizer_model)
        eos_id = _get_eos_token_id(enc, self.tokenizer_model) if self.add_eos else None

        files = self._shard_for_worker(self._files())
        if not files:
            return

        token_buffer: List[int] = []

        for file in files:
            dataset = ds.dataset([file], format="parquet")
            col = self.text_col or choose_string_column(dataset)
            scanner = dataset.scanner(batch_size=self.batch_rows)

            for rb in scanner.to_batches():
                arr = rb[col]
                # Accumulate tokens for this batch of rows
                for raw in arr.to_pylist():
                    if raw is None:
                        continue
                    text = str(raw).strip()
                    if not text:
                        continue
                    if not is_english_text(text, mode=self.lang_mode, ascii_threshold=self.ascii_threshold):
                        continue
                    try:
                        toks = enc.encode_ordinary(text)
                    except Exception:
                        toks = enc.encode(text)
                    if toks:
                        token_buffer.extend(toks)
                        if eos_id is not None:
                            token_buffer.append(eos_id)

                # As tokens accumulate, yield sequences
                for seq in self._yield_sequences(token_buffer):
                    # Each item from this IterableDataset is one 1D seq of length block_size+1
                    yield seq

            # small compaction at file boundary
            # (keep at most block_size tokens to continue across files)
            if len(token_buffer) > self.block_size:
                token_buffer[:] = token_buffer[-self.block_size:]

# --- NEW: simple collate that turns seq->(x,y) --------------------------
def shift_collate(batch: List[torch.Tensor]):
    """
    batch: list of tensors shape (block_size+1,)
    returns:
        X: (B, block_size)
        Y: (B, block_size)
    """
    seqs = torch.stack(batch, dim=0)  # (B, L+1)
    return seqs[:, :-1].contiguous(), seqs[:, 1:].contiguous()


GPT2 Training Sliding Window with Overlap

In [8]:
## -- Utility Functions ---------------------------------------------------
# -- Getting Size of Vocab Function ------------------------------------------
def get_vocab_size(encoding_name: str) -> int:
    """
    Return the vocabulary size for a given encoding in tiktoken.

    Args:
        encoding_name: Name of the encoding (e.g. "gpt2", "cl100k_base", etc.)

    Returns:
        The total number of tokens in that encoding’s vocabulary.
    """
    encoding = tiktoken.get_encoding(encoding_name)
    return encoding.n_vocab


In [9]:


@torch.no_grad()
def estimate_loss_from_loaders(
    model: nn.Module,
    *,
    train_loader,
    val_loader,
    device,                # torch.device or str accepted; used only for .to(...)
    device_type: str,      # "cuda" or "cpu" - used by autocast
    autocast_dtype,
    eval_iters: int,
    max_eval_iters: int = 50,
):
    """
    Robust evaluation helper that computes mean loss over `eval_iters` batches
    from both train and val loaders.

    Notes / defensive behavior:
      - Accepts keyword-only loader args to avoid accidental reordering.
      - Ensures `eval_iters` at least 1 and clamps by max_eval_iters.
      - Handles StopIteration gracefully if a loader is shorter than eval_iters.
      - Temporarily disables gradient-checkpointing if the model supports it.
      - Uses `torch.autocast(device_type=..., dtype=...)` for the forward pass.
    Returns:
      dict with keys: {"train": float_mean_loss, "val": float_mean_loss}
    """
    model_was_training = model.training
    model.eval()

    had_checkpoint = False
    touched = []

    
    # If modules have `use_checkpoint` field, temporarily disable them
    for m in model.modules():
        if hasattr(m, "use_checkpoint"):
            touched.append((m, getattr(m, "use_checkpoint")))
            setattr(m, "use_checkpoint", False)
            had_checkpoint = True

    eval_iters = int(max(1, min(max_eval_iters, eval_iters)))

    totals = {"train": 0.0, "val": 0.0}
    counts = {"train": 0, "val": 0}

    # ensure device is usable for .to(...)
    dev = device if isinstance(device, torch.device) else torch.device(device)

    # iterate both loaders explicitly and defensively
    for name, loader in (("train", train_loader), ("val", val_loader)):
        it = iter(loader)
        for _ in range(eval_iters):
            try:
                X, Y = next(it)
            except StopIteration:
                # loader exhausted early; stop consuming this loader
                break
            X = X.to(dev, non_blocking=True)
            Y = Y.to(dev, non_blocking=True)
            # use explicit device_type for autocast
            with torch.autocast(device_type=device_type, dtype=autocast_dtype):
                _, loss = model(X, Y)
            totals[name] += float(loss.item())
            counts[name] += 1

    # Restore any per-module checkpoint flags
    for m, old in touched:
        setattr(m, "use_checkpoint", old)

    if model_was_training:
        model.train()

    # Compute means (avoid zero-divide)
    out = {}
    for k in ("train", "val"):
        if counts[k] == 0:
            out[k] = float("nan")
        else:
            out[k] = totals[k] / counts[k]
    return out




In [10]:


# -- Checkpoint Saving -------------------------------------------------
def save_checkpoint(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    loss: float,
    batches_seen: List[int],
    train_losses: List[float],
    val_losses: List[float],
    file_path: str,
    scaler: torch.cuda.amp.GradScaler = None, # Optional scaler argument
    warmup_scheduler: torch.optim.lr_scheduler.LRScheduler = None,  # Optional warmup argument
    plateau_scheduler: torch.optim.lr_scheduler.LRScheduler = None  # Optional plateau argument
) -> None:
    """ 
    Save model, optimizer, scaler (optional), and training history to disk for later resumption and plotting.

    Args:
        model (torch.nn.Module): Trained GPTLanguageModel.
        optimizer (torch.optim.Optimizer): Associated optimizer (e.g., AdamW).
        epoch (int): Current epoch or step number.
        loss (float): Latest loss value.
        batches_seen (List[int]): List of batch-indices at which evals were performed.
        train_losses (List[float]): Recorded training losses at each eval point.
        val_losses (List[float]): Recorded validation losses at each eval point.
        file_path (str): File path for checkpoint.
        scaler (torch.cuda.amp.GradScaler, optional): GradScaler for mixed precision training.

    Notes:
        - Saves a dict with keys:
          * 'epoch', 'model_state_dict', 'optimizer_state_dict', 'loss'
          * 'history': contains 'batches_seen', 'train_losses', 'val_losses'
          * 'scaler_state_dict' (if scaler is provided)
        - Later, loading this checkpoint allows both exact model/optimizer restoration
          and re-plotting of the entire train/val loss curve.
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
        'history': {
            'batches_seen': batches_seen,
            'train_losses': train_losses,
            'val_losses': val_losses,
        }
    }
    if scaler is not None:
        checkpoint['scaler_state_dict'] = scaler.state_dict()
    if warmup_scheduler is not None:
        checkpoint['warmup_state_dict'] = warmup_scheduler.state_dict()
    if plateau_scheduler is not None:
        checkpoint['plateau_scheduler_state_dict'] = plateau_scheduler.state_dict()
    torch.save(checkpoint, file_path)


def get_total_params_gpt2(model: torch.nn.Module) -> int:
    """
    Compute the total number of *unique* trainable parameters in a GPT-2 model,
    correctly accounting for weight tying between the input embeddings and the
    output language modeling head.

    GPT-2 ties the weights of its input embedding matrix and the output projection
    (lm_head).  Although these weights appear twice in the model's parameter list,
    they should only be counted once when reporting the total number of parameters.

    Args:
        model (torch.nn.Module):
            A GPT-2 model instance (e.g. from Hugging Face's transformers library),
            which must have an attribute `lm_head` representing the output projection
            layer.

    Returns:
        int: The total number of unique parameters in the GPT-2 model, with the
             duplicated `lm_head` parameters subtracted out.
    """
    # 1) Count every parameter in the model (embeddings, transformer blocks, lm_head, etc.)
    total_params = sum(p.numel() for p in model.parameters())

    # 2) Count only the parameters in the output head (lm_head).
    #    These share weights with the input embedding matrix under weight-tying.
    lm_head_params = sum(p.numel() for p in model.lm_head.parameters())

    # 3) Subtract the lm_head params once so they're not double-counted,
    #    yielding the true GPT-2 parameter count.
    unique_gpt2_params = total_params - lm_head_params

    return unique_gpt2_params

def get_total_model_size_mb(model: torch.nn.Module) -> int:
    total_params = sum(p.numel() for p in model.parameters())
    total_size_bytes = total_params * 4 #A
    total_size_mb = total_size_bytes / (1024 * 1024) #B
    return total_size_mb

In [11]:

"""
Streaming pre-training loop for a GPT-style autoregressive language model.

High-level flow
---------------
1. Data are streamed from Parquet shards → tokenized on-the-fly → overlapping
   sliding windows (stride < block_size) to maximize data efficiency.
2. Mixed-precision (`bfloat16`) and gradient accumulation are used to fit large
   models and long sequences on modest GPUs.
3. Training is measured in **effective steps** (optimizer updates) rather than
   micro-batches for intuitive scheduling of evals, checkpoints, LR warm-up, etc.
4. Early stopping, LR-scheduling on plateau, NaN guards, and deterministic
   checkpointing are all built-in.

Key symbols
-----------
effective_step       number of times the optimizer has actually stepped
batches_processed    raw mini-batches consumed (micro-steps)
grads_accum_steps    how many micro-batches are averaged into one effective step
eval_interval        effective steps between validation runs
save_interval        effective steps between checkpoints
warmup_steps         effective steps over which LR rises from 0 → base_lr

Implementation notes
--------------------
- optimizer.zero_grad is called once per effective step only.
- Loss is **mean-reduced** across the accumulation window via
  `loss = loss / grads_accum_steps`.
- tqdm bar updates on every effective step; logs and filenames use the same unit.
- ReduceLROnPlateau is stepped after **every** validation (not only when the
  interval doubles) to guarantee LR decay when loss plateaus.
- Warm-up uses effective_step, so `warmup_steps=2000` means 2000 optimizer updates.
"""

# -- Logging Configuration ----------------------------------------------
run_id = int(time.time())

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S", 
    encoding='utf-8'
)
logger = logging.getLogger(__name__)

# Ensure logs directory exists
LOG_DIR = os.path.join("logs")
if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)

# Create file handler
log_file = os.path.join(LOG_DIR, f"train_{run_id}.log")
file_handler = logging.FileHandler(log_file, mode='w', encoding='utf-8')
file_handler.setLevel(logging.DEBUG)

# Optional: use same format as console
formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%H:%M:%S")
file_handler.setFormatter(formatter)

# Add handler to logger
logger.addHandler(file_handler)
logger.info(f"Logging to file: {log_file}")

# -------------------------
# Device / autocast / precision setup
# -------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
logger.info(f"Using device: {device}")

# Normalize device_type string for autocast
# `device` may be a torch.device or string like 'cuda' or 'cpu' -- normalize:
if isinstance(device, torch.device):
    device_type = 'cuda' if device.type == 'cuda' else 'cpu'
else:
    device_type = 'cuda' if str(device).startswith('cuda') else 'cpu'
autocast_dtype = torch.bfloat16 if device_type == 'cuda' else torch.float32

# Matmul precision setting (GPU)
try:
    torch.set_float32_matmul_precision('high')
    print("torch.set_float32_matmul_precision('high')")
except Exception:
    # ignore if not available
    pass

seed = 1337
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if device == 'cuda':
    torch.cuda.manual_seed_all(seed)



# -- Load Tokenizer -----------------------------------------------------
encoding_name = "gpt2"
tokenizer = tiktoken.get_encoding(encoding_name)
vocab_size = get_vocab_size(encoding_name) 
while(vocab_size%64 != 0): vocab_size+=1 # override with larger value that is divisable by 64 
logger.info(f"Vocab size: {vocab_size}")

# -- Model Configuration ------------------------------------------------
torch.manual_seed(1337)
block_size = 1024
n_embd = 256 # 768 1024 1280
n_head = 2 # 12 16 20
n_layer = 2 # 12 24 36
dropout = 0.1
batch_size = 1


# create an instance of GPTLanguageModel class
model = GPTLanguageModel(
    vocab_size=vocab_size, 
    block_size=block_size,
    n_embd=n_embd,
    n_head=n_head,
    n_layer=n_layer,
    dropout=dropout,
    device=device
).to(device)
#logger.info(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
logger.info(f"Model parameters: {get_total_params_gpt2(model)/1e6:.2f}M")
logger.info(f"Total size of the model: {get_total_model_size_mb(model):.2f} MB")


# -- Load Data ----------------------------------------------------------
INPUT_DIR = "datasets/100BT"
PARQUET_GLOB = "*.parquet"
all_files = sorted(glob(os.path.join(INPUT_DIR, PARQUET_GLOB)))
assert len(all_files) > 0, "No parquet files found."

# -- Train/Val Split -----------------------------------------------------
# allocating 90% for training and 10% for validation
# simple file-level split (deterministic)
rng = random.Random(1337)
rng.shuffle(all_files)
split_at = max(1, int(0.9 * len(all_files)))
train_files = all_files[:split_at]
logger.info(f"Training files total -> {len(train_files)}: {train_files}")
val_files   = all_files[split_at:] or all_files[-1:]
logger.info(f"Validation files total -> {len(val_files)}: {val_files}")




# -- Prepare Overlapping Windows -----------------------------------
# Desired fraction of overlap between successive windows
overlap_frac = 0.0625   # 6.25% 
#overlap_frac = 0.5     # 50% 
# Compute the stride (how far the window moves each time)
stride = int(block_size * (1 - overlap_frac))  

num_workers = min(os.cpu_count() or 1, 8)    # tune
num_workers = 0                             # zero for running in ipynb notebook
prefetch_factor = 4                          # tune with GPU util

train_ds = ParquetCausalIterable(
    files=train_files,
    tokenizer_model="gpt2",
    add_eos=True,
    block_size=block_size,
    stride=stride,
    lang_mode="skip",          # or "accuracy"/"ascii" like in your extractor
    ascii_threshold=0.5,
    shuffle_files=True,
    seed=1337,
)

val_ds = ParquetCausalIterable(
    files=val_files,
    tokenizer_model="gpt2",
    add_eos=True,
    block_size=block_size,
    stride=stride,
    lang_mode="skip",
    ascii_threshold=0.5,
    shuffle_files=True,       
    seed=2024,
)

pin = (device == "cuda")
train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=False,                 # IterableDataset must be shuffle=False
    num_workers=num_workers,
    pin_memory=pin,
    prefetch_factor=prefetch_factor if num_workers > 0 else None,
    persistent_workers=(num_workers > 0),
    collate_fn=shift_collate,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin,
    prefetch_factor=prefetch_factor if num_workers > 0 else None,
    persistent_workers=(num_workers > 0),
    collate_fn=shift_collate,
    drop_last=True,
)


# -- Optimizer & Training Setup -----------------------------------------
grads_accum_steps = 8
initial_eval_interval = 32
max_eval_interval = 1000
save_interval = max_eval_interval * 10
eval_interval = initial_eval_interval
val_loss_threshold = 3.0  # stop if val loss exceeds threshold x train loss
warmup_steps = 2000
patience_limit = 10 # stop if no improvement in val loss after patience_limit checks
step = 0


learning_rate = 5e-4
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=learning_rate,
    weight_decay=0.01  # Regularization
)
warmup = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda effective_step: min(1.0, effective_step / warmup_steps)
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,   # new_LR = LR*factor
    patience=3,   # number of evals with no improvement to wait before appying new_LR
    min_lr = 1e-6
)

# generate from the model before training
input_tokens = tokenizer.encode("I like apple juice, I drink it")
input_tokens = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    output = model.generate(input_tokens=input_tokens, max_new_tokens=30)
# Safe decoding: ignore tokens outside tokenizer's known range
decoded = tokenizer.decode([t for t in output[0].tolist() if t < tokenizer.n_vocab])
logger.info(f"Model output: \n{decoded}")
model.train()


04:21:16 [INFO] Logging to file: logs\train_1757125276.log
04:21:16 [INFO] Using device: cuda


torch.set_float32_matmul_precision('high')


04:21:16 [INFO] Vocab size: 50304
04:21:16 [INFO] Model parameters: 2.10M
04:21:16 [INFO] Total size of the model: 57.32 MB
04:21:16 [INFO] Training files total -> 126: ['datasets/100BT\\000_00000.parquet', 'datasets/100BT\\012_00008.parquet', 'datasets/100BT\\007_00001.parquet', 'datasets/100BT\\001_00008.parquet', 'datasets/100BT\\008_00007.parquet', 'datasets/100BT\\000_00007.parquet', 'datasets/100BT\\010_00005.parquet', 'datasets/100BT\\006_00002.parquet', 'datasets/100BT\\003_00002.parquet', 'datasets/100BT\\013_00004.parquet', 'datasets/100BT\\002_00004.parquet', 'datasets/100BT\\001_00007.parquet', 'datasets/100BT\\013_00005.parquet', 'datasets/100BT\\006_00008.parquet', 'datasets/100BT\\006_00000.parquet', 'datasets/100BT\\012_00004.parquet', 'datasets/100BT\\012_00006.parquet', 'datasets/100BT\\005_00000.parquet', 'datasets/100BT\\005_00009.parquet', 'datasets/100BT\\001_00004.parquet', 'datasets/100BT\\003_00008.parquet', 'datasets/100BT\\002_00006.parquet', 'datasets/100BT\

GPTLanguageModel(
  (token_embedding_table): Embedding(50304, 256)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (qkv_proj): Linear(in_features=256, out_features=768, bias=False)
        (out_proj): Linear(in_features=256, out_features=256, bias=False)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ffwd): FeedForward(
        (w1): Linear(in_features=256, out_features=1024, bias=False)
        (w2): Linear(in_features=256, out_features=1024, bias=False)
        (w3): Linear(in_features=1024, out_features=256, bias=False)
        (drop): Dropout(p=0.1, inplace=False)
      )
      (ln1): RMSNorm()
      (ln2): RMSNorm()
    )
    (1): Block(
      (sa): MultiHeadAttention(
        (qkv_proj): Linear(in_features=256, out_features=768, bias=False)
        (out_proj): Linear(in_features=256, out_features=256, bias=False)
        (attn_dropou

In [12]:
print(show_vram_usage())



Used: 865 MB, Free: 11422 MB, Total: 12288 MB


In [ ]:
# -------------------------
# Prepare directories
# -------------------------
pre_training_dir = os.path.join('outputs', 'output_gpt_gemma3_hybrid_v1', 'pre_training', f'run_{run_id}')
os.makedirs(pre_training_dir, exist_ok=True)

# -------------------------
# GradScaler setup
# -------------------------
if device_type == "cuda":
    # BF16 is supported natively on Ampere+ GPUs, so GradScaler is not always necessary.
    # However, if you mix FP16 precision, enable it for stability.
    scaler = GradScaler(enabled=(autocast_dtype == torch.float16))
else:
    # No-op on CPU
    class DummyScaler:
        def scale(self, loss): return loss
        def step(self, optimizer): optimizer.step()
        def unscale_(self, optimizer): return
        def update(self): return
        def state_dict(self): return {}
        def load_state_dict(self, _: dict): return
    scaler = DummyScaler()

# -------------------------
# Tracking metrics 
# --------------------------
batches_processed = 0        # raw mini-batches seen (micro-steps)
effective_step = 0           # full optimizer steps
last_grad_norm = 0.0
train_losses, val_losses, batches_seen = [], [], []
val_loss = None
early_stop = False
t0 = time.time()
best_val_loss = float('inf')
patience_counter = 0

# decide how many *effective* steps you want (or math.inf for endless streaming)
max_effective_steps = math.inf   # change to taste

# -------------------------
# Prepare tqdm bar
# -------------------------
pbar_total = None if math.isinf(max_effective_steps) else int(max_effective_steps)
pbar = tqdm(total=pbar_total, desc="Training", unit="step", leave=True)

# initialize a consistent postfix dictionary
postfix = {
    "loss": "-",
    "grad": "-",
    "train": "-",
    "val": "-",
    "lr": "-",
    "t/step": "-",
    "t/eval": "-"
}


# -------------------------
# Start training loop
# -------------------------
# Streaming training loop (no epochs) 
train_iter = iter(train_loader)  # fresh iterator
# we now zero once *before* the very first micro-batch
optimizer.zero_grad(set_to_none=True)

try:
    while not early_stop and effective_step < max_effective_steps:
        micro_start = time.time()          # <── start timer

        # fetch next micro-batch (streaming)
        try:
            xb, yb = next(train_iter)
        except StopIteration:
            # reset iterator if streaming dataset exhausted
            train_iter = iter(train_loader)
            xb, yb = next(train_iter)
            logger.info(f"Training iterator reset")
            
        # move to device (non_blocking requires pinned memory)
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # forward with autocast — handle dtype/device gracefully
        # Precompute a display_loss before backward (to avoid issues with scaled tensors)
        with autocast(device_type=device_type, dtype=autocast_dtype):
            _, loss = model(xb, yb)
            # scale down the loss for accumulation (important)
            loss = loss / grads_accum_steps
            display_loss = float((loss * grads_accum_steps).detach().cpu())

        if device_type == 'cuda': torch.cuda.synchronize()
        if torch.isnan(loss).any():
            logger.error("NaN loss at micro-batch %d (batches_processed=%d)", batches_processed, batches_processed)
            # Save checkpoint for debugging (optional)
            try:
                ckpt_path = os.path.join(pre_training_dir, f"checkpoint_nan_at_micro_{batches_processed}.pth")
                save_checkpoint(
                    model=model,
                    optimizer=optimizer,
                    epoch=effective_step,
                    loss=loss.item() if hasattr(loss, 'item') else float('nan'),
                    batches_seen=batches_processed,
                    train_losses=train_losses,
                    val_losses=val_losses,
                    file_path=ckpt_path,
                    scaler=scaler,
                    warmup_scheduler=warmup,
                    plateau_scheduler=scheduler,
                )
                logger.info("Saved NaN checkpoint to %s", ckpt_path)
            except Exception as e:
                logger.exception("Failed to save NaN checkpoint: %s", e)

            early_stop = True
            break
                    
        # backward
        scaler.scale(loss).backward()
        batches_processed += 1


        # When we reached accumulation boundary -> optimizer step + scheduler + logging + checkpoints
        if batches_processed % grads_accum_steps == 0:
            # Unscale before clipping
            scaler.unscale_(optimizer)
            # Clip grads (returns total_norm)
            last_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            # Optimizer + scaler step
            scaler.step(optimizer)
            scaler.update()
            # zero grads (set_to_none for slight perf improvement)
            optimizer.zero_grad(set_to_none=True)

            effective_step += 1

            # update postfix with micro-batch info
            postfix.update({
                "loss": f"{(loss*grads_accum_steps).item():.4f}",
                "grad": f"{last_grad_norm:.2f}",
                "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
                "t/step": f"{time.time() - micro_start:.3f}s"
            })
            pbar.update(1)
            pbar.set_postfix(postfix)

            # LR warmup scheduler (if provided)
            if effective_step < warmup_steps:
                try:
                    warmup.step()
                    postfix["lr"] = f"{optimizer.param_groups[0]['lr']:.2e}"
                    logger.debug(f"Warmup step {effective_step}/{warmup_steps} - LR: {postfix['lr']}")
                except Exception:
                    # fail gracefully if warmup not configured properly
                    logger.debug("Warmup step failed or warmup scheduler missing")

            # Periodic checkpoint: save at multiples of save_interval including the first
            if effective_step % save_interval == 0 and effective_step > 1:
                ckpt_path = os.path.join(pre_training_dir, f"checkpoint_step_{effective_step}.pth")
                save_checkpoint(
                    model=model,
                    optimizer=optimizer,
                    epoch=effective_step,
                    loss=loss.item() if hasattr(loss, 'item') else None,
                    batches_seen=batches_processed,
                    train_losses=train_losses,
                    val_losses=val_losses,
                    file_path=ckpt_path,
                    scaler=scaler,
                    warmup_scheduler=warmup,
                    plateau_scheduler=scheduler,
                )
                logger.info(f"Saved periodic checkpoint @ step {effective_step}")

            # Evaluation & logging: run every eval_interval effective steps
            if effective_step % eval_interval == 0 and effective_step > 1:
                dt = time.time() - t0
                t0 = time.time()

                # 1. Run validation
                # Model into eval mode for metric estimation
                model.eval()
                with torch.no_grad():                 
                    metrics = estimate_loss_from_loaders(
                        model,
                        train_loader=train_loader,
                        val_loader=val_loader,
                        device=device,
                        device_type=device_type,        # pass explicit device_type (string)
                        autocast_dtype=autocast_dtype,
                        eval_iters=eval_interval,
                    )
                model.train()

                # store metrics (expecting dict with 'train' and 'val')
                train_loss = metrics.get('train', float('nan'))
                val_loss = metrics.get('val', float('nan'))
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                batches_seen.append(batches_processed)

                # 2. Scheduler
                # scheduler step: many schedulers expect val_loss or epoch; using val_loss for ReduceLROnPlateau
                try:
                    scheduler.step(val_loss)
                except Exception:
                    # fallback: if scheduler expects step-per-epoch, call differently / ignore.
                    logger.debug("Scheduler.step(val_loss) failed; check scheduler type/usage")

                # 3. Update tqdm bar
                # update postfix with evaluation info
                postfix.update({
                    "train": f"{train_loss:.4f}",
                    "val": f"{val_loss:.4f}",
                    "t/eval": f"{dt:.1f}s"
                })
                pbar.set_postfix(postfix)

                # quick sample generation for sanity check (keep it short)
                model.eval()
                with torch.no_grad():
                    input_tokens = tokenizer.encode("I like apple juice, I drink it")
                    input_tokens = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)
                    output = model.generate(input_tokens=input_tokens, max_new_tokens=30)
                    decoded = tokenizer.decode([t for t in output[0].tolist() if t < tokenizer.n_vocab])
                    logger.info(f"Sample: {decoded}")
                model.train()
                
                # Increase eval interval if desired:
                eval_interval = min(eval_interval * 2, max_eval_interval)

                            # Divergence guard: stop if validation explode relative to training
                if train_loss and not math.isnan(train_loss):
                    if val_loss > val_loss_threshold * train_loss:
                        logger.warning(f"Validation loss {val_loss:.4f} exceeds {val_loss_threshold}x training loss {train_loss:.4f}. Stopping.")
                        save_checkpoint(
                            model=model,
                            optimizer=optimizer,
                            epoch=effective_step,
                            loss=loss.item() if hasattr(loss, 'item') else None,
                            batches_seen=batches_processed,
                            train_losses=train_losses,
                            val_losses=val_losses,
                            file_path=os.path.join(pre_training_dir, f"checkpoint_divergence_step_{effective_step}.pth"),
                            scaler=scaler,
                            warmup_scheduler=warmup,
                            plateau_scheduler=scheduler,
                        )
                        early_stop = True
                        break

                # Patience early stopping
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1
                    logger.info(f"No val_loss improvement {patience_counter}/{patience_limit}")
                    if patience_counter >= patience_limit:
                        logger.warning("Early stopping triggered by patience limit")
                        save_checkpoint(
                            model=model,
                            optimizer=optimizer,
                            epoch=effective_step,
                            loss=loss.item() if hasattr(loss, 'item') else None,
                            batches_seen=batches_processed,
                            train_losses=train_losses,
                            val_losses=val_losses,
                            file_path=os.path.join(pre_training_dir, f"checkpoint_patience_step_{effective_step}.pth"),
                            scaler=scaler,
                            warmup_scheduler=warmup,
                            plateau_scheduler=scheduler,
                        )
                        early_stop = True
                        break

    # -- Final log ------------------------------------------------------
    if early_stop:
        logger.info("Training stopped by early stopping criteria.")
    else:
        logger.info("Training stopped manually or completed streaming run.")

except KeyboardInterrupt:
    logger.warning("KeyboardInterrupt — saving checkpoint before exit")
    save_checkpoint(
        model=model,
        optimizer=optimizer,
        epoch=effective_step,
        loss=loss.item() if hasattr(loss, 'item') else None,
        batches_seen=batches_processed,
        train_losses=train_losses,
        val_losses=val_losses,
        file_path=os.path.join(pre_training_dir, "checkpoint_interrupt.pth"),
        scaler=scaler,
        warmup_scheduler=warmup,
        plateau_scheduler=scheduler,
    )
    raise
except Exception as e:
    logger.exception("Unexpected exception. Saving checkpoint.")
    save_checkpoint(
        model=model,
        optimizer=optimizer,
        epoch=effective_step,
        loss=loss.item() if hasattr(loss, 'item') else None,
        batches_seen=batches_processed,
        train_losses=train_losses,
        val_losses=val_losses,
        file_path=os.path.join(pre_training_dir, "checkpoint_exception.pth"),
        scaler=scaler,
        warmup_scheduler=warmup,
        plateau_scheduler=scheduler,
    )
    raise
finally:
    pbar.close()


Training: 32step [00:08,  6.39step/s, loss=10.7747, grad=2.06, train=10.7632, val=10.7572, lr=8.00e-06, t/step=0.031s, t/eval=5.9s]04:21:26 [INFO] Sample: I like apple juice, I drink itazingnoon?????BrownNation underest squemission phenotype statueatron susceptibilitySimon306 stre Pokémon widgets SpecorrectKnownThroughoutJa551 incentiv grave sideline dreamMut trust Debor
Training: 64step [00:18,  4.57step/s, loss=10.5029, grad=1.71, train=10.4644, val=10.4738, lr=1.60e-05, t/step=0.044s, t/eval=9.7s]04:21:36 [INFO] Sample: I like apple juice, I drink it logo catapult kan Linganth revolutions meticulousighed hitherto potassium militants language Charlie BeansTileivalryobjects////////////////itimePIollaOSTedly organism hatched brakingyden doorsiated symbolism
Training: 128step [00:36,  3.09step/s, loss=10.0190, grad=1.46, train=9.9054, val=9.9253, lr=3.20e-05, t/step=0.044s, t/eval=17.9s] 04:21:54 [INFO] Sample: I like apple juice, I drink it (?,iter Ji senior� Magesicolava documenting C

In [ ]:
# -- Plot training and validation loss vs batch number ------------------------------------------------------
import matplotlib.pyplot as plt

def plot_train_val_loss(batches_seen, train_losses, val_losses):
    logger.info("Generating plot: training and validation loss vs batch number.")

    fig, ax1 = plt.subplots(figsize=(15,6))

    # Plot training and validation loss against epochs
    ax1.plot(batches_seen, train_losses, label="Train Loss")
    ax1.plot(batches_seen, val_losses, linestyle="-.", label="Val Loss")
    ax1.set_xlabel("Batch Number")
    #ax1.set_ylim(0)
    ax1.set_ylabel("Loss")
    ax1.set_title("Train & Validation Loss")
    ax1.legend(loc="upper right")
    # --- Add Subgrid (Minor Grid Lines) ---
    ax1.minorticks_on()  # Enable minor ticks
    # Major grid (main grid lines)
    ax1.grid(True, which='major', linestyle='-', linewidth=0.5, alpha=0.8)
    # Minor grid (subgrid lines)
    ax1.grid(True, which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
    ax1.grid(True)

    #fig.tight_layout()  # Adjust layout to make room
    plt.xticks(rotation=45)
    plt.savefig(f"loss-plot-{run_id}.pdf")
    plt.show()

plot_train_val_loss(batches_seen, train_losses, val_losses)


In [ ]:
# generate from the model
input_tokens = tokenizer.encode("I like apple juice - I drink it")
input_tokens = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model.advanced_generation(input_tokens=input_tokens, max_new_tokens=40, temperature=1.3, top_k=20)

# Safe decoding: ignore tokens outside tokenizer's known range
decoded = tokenizer.decode([t for t in output[0].tolist() if t < tokenizer.n_vocab])
logger.info(f"Model output: \n{decoded}")

In [ ]:
# -- load model form checkpoint----------------------------------------------------

# 1) Re‑instantiate everything with the *same* hyperparameters
vocab_size = 50304
block_size = 1024
n_embd = 1024 # 768 1024 1280
n_head = 16 # 12 16 20
n_layer = 24 # 12 24 36
dropout = 0.1
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GPTLanguageModel(
    vocab_size=vocab_size,
    block_size=block_size,
    n_embd=n_embd,
    n_head=n_head,
    n_layer=n_layer,
    dropout=dropout,
    device=device
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scaler = GradScaler(device, enabled=(device == 'cuda'))

# 2) Load tokenizer as before
encoding_name = "gpt2"
tokenizer = tiktoken.get_encoding(encoding_name)

# 3) Point to your checkpoint
ckpt_path = r"outputs\output_v17\pre_training\run_1756251457\checkpoint_step_100000.pth"
assert os.path.isfile(ckpt_path), f"Checkpoint not found at {ckpt_path}"

# 4) Load the checkpoint
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
optimizer.load_state_dict(ckpt['optimizer_state_dict'])

# # If you saved the scaler state dict, restore it
# if 'scaler_state_dict' in ckpt:
#     scaler.load_state_dict(ckpt['scaler_state_dict'])

# # When loading checkpoint:
# if 'warmup_scheduler_state_dict' in checkpoint:
#     warmup_scheduler.load_state_dict(checkpoint['warmup_scheduler_state_dict'])
# if 'plateau_scheduler_state_dict' in checkpoint:
#     scheduler.load_state_dict(checkpoint['plateau_scheduler_state_dict'])

# Load plot history
history = ckpt['history']
batches_seen = history['batches_seen']
train_losses = history['train_losses']
val_losses   = history['val_losses']

start_step = ckpt.get('epoch', None)
print(f"Loaded checkpoint from step {start_step}. Resuming from there.")

# 5) (Optional) Set model to eval or train
# For continued training:
model.train()
# For inference only:
# model.eval()

# 6) Example inference to verify it works:
# generate from the model
input_tokens = tokenizer.encode("I like apple juice - I drink it")
input_tokens = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model.advanced_generation(input_tokens=input_tokens, max_new_tokens=50, temperature=1.7, top_k=10)

# Safe decoding: ignore tokens outside tokenizer's known range
decoded = tokenizer.decode([t for t in output[0].tolist() if t < tokenizer.n_vocab])
logger.info(f"Model output: \n{decoded}")